# is_lotte_related 이진 분류기 학습 (Phase 5 — 4차 재학습)

**모델:** `klue/roberta-large` fine-tuning
**분류:** 이진 (BCEWithLogitsLoss, num_labels=1)
**입력:** title (seq-A) + description_snippet (seq-B)
**max_length:** 256
**목표:** val recall ≥ 0.97, precision ≥ 0.90
**예상 소요:** T4 GPU 기준 약 15~25분

## 3차 → 4차 재학습 변경 사항

| 항목 | 3차 | 4차 |
|------|-----|-----|
| labeled_titles 행 수 | 3,181행 | 3,181행 |
| labeled_titles True | 1,772건 | 1,760건 |
| labeled_titles False | 1,409건 | 1,668건 |
| labeled_players 행 수 | 6,208행 | 6,708행 |
| labeled_players False | 222건 | 722건 |
| 합산 False (FILTER 전) | ~1,631건 | ~2,390건 |
| 데이터 품질 | 3차 FP 패턴 일부 혼입 | 타팀 주체 FP 패턴 교정 완료 |

**주요 변경:**
- 오분류 32건 True→False 교정 (`corrected_false_otherteam_subject`)
  - 라운드업형 4건: 키움·삼성 스윗 등 여러 경기 결과 뿃음 기사 (롯데는 snippet에만 등장)
  - 타팀 선수/이슈 분석형 28건: KIA 카스트로 부상, 두산 외인, 한화 트레이드 등
- review_lotte_related.csv corrected_value 교정
  - ID 124, 155 → False (타팀 분석/라운드업, 롯데는 배경 언급)
  - ID 228 → True 유지 (사직구장 롯데전이 기사 핵심 맥락)
- collect_lotte_unrelated.py LOTTE_OPPONENT_KEYWORDS 확장
  - 라운드업 수집용: 비-롯데 팀 조합 ("두산 삼성", "KIA LG" 등)
  - 타팀 이슈 수집용: "KIA 외인", "KIA 부상", "두산 대체 외인" 등
- eval_lotte_related.py: FILTER_GPT_MISSING 적용, 스모크 테스트 3차 Colab 케이스 동기화

**사전 준비**
- 런타임 유형: T4 GPU (런타임 → 런타임 유형 변경)
- 업로드 파일: `labeled_titles.csv`, `labeled_players.csv`

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('[WARN] GPU 없음 — CPU 학습 시 수 시간 소요')

In [ ]:
!pip install -q transformers torch scikit-learn pandas numpy

In [ ]:
# ── 하이퍼파라미터 (필요 시 수정) ─────────────────────────────────────────────
# 데이터 현황 (2026-06-05 4차 재학습 기준):
#   labeled_titles.csv   3,434행  True=1,760  False=1,668
#     └ 4차 변경: 오분류 32건 True→False 교정, 재수집 후 오수집 11건 False→True 재교정
#               confidence_note: corrected_false_otherteam_subject / corrected_true_lotte_game
#   labeled_players.csv  6,708행  True=5,986  False=722
#     └ False 출처: hard_negative 722건 (야구맥락 또는 롯데브랜드 보장)
#   합산 (dedup 전)      10,142행  True=7,746 (76.4%)  False=2,390 (23.6%)
#   FILTER_GPT_MISSING=True 적용 시 gpt_missing_index 제거 → False 소폭 감소
#   pos_weight raw = 2,390/7,746 ≈ 0.309 → clamp(0.3) 이하, 그대로 적용
#
# 3차 대비 변경 사항:
#   - 타팀 주체 기사 FP 패턴 32건 False 교정 (eval 스모크 테스트로 발견)
#     · 라운드업형 4건: 키움·삼성 스윗 등 여러 경기 묶음 (롯데 snippet에만 등장)
#     · 타팀 분석형 28건: KIA 카스트로 부상, 두산 외인, 한화 트레이드 등
#   - labeled_players.csv hard negative 500건 추가 (False 222→722)
#   - collect_lotte_unrelated.py 키워드 확장 (라운드업·분석형 패턴)
#   - eval_lotte_related.py FILTER_GPT_MISSING 동기화, 스모크 테스트 갱신

PRETRAINED     = 'klue/roberta-large'
MAX_LENGTH     = 256
BATCH_SIZE     = 8
GRAD_ACCUM     = 2          # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
EPOCHS         = 7
LR             = 3e-5
WARMUP_RATIO   = 0.1
SEED           = 42
VAL_SPLIT      = 0.15
RECALL_TARGET  = 0.97
DEFAULT_THRESH = 0.40
SNIPPET_LEN    = 300
DATA_DIR       = '/content/data'
OUTPUT_DIR     = '/content/lotte_related_model'

# False 샘플 품질 필터:
#   gpt_missing_index 행 제거 — GPT 배치 처리 중 누락된 인덱스를 강제로 False 레이블한 불확실 행
#   hard_negative_player_context 행은 신뢰 가능 (labeled_players, 야구맥락 보장)
FILTER_GPT_MISSING = True

VALID_POS = {'true', '1', 'yes'}
VALID_NEG = {'false', '0', 'no'}
print('Config 설정 완료')

In [ ]:
import os
from google.colab import files

os.makedirs(DATA_DIR, exist_ok=True)
print('labeled_titles.csv 와 labeled_players.csv 를 선택하세요.')
uploaded = files.upload()
for fname, content in uploaded.items():
    dst = f'{DATA_DIR}/{os.path.basename(fname)}'
    with open(dst, 'wb') as f:
        f.write(content)
    print(f'저장: {dst}  ({len(content):,} bytes)')

In [ ]:
import pandas as pd

total_pos, total_neg = 0, 0
for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        print(f'[MISSING] {fname}')
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    raw = df['is_lotte_related'].astype(str).str.strip().str.lower()
    pos = raw.isin(VALID_POS).sum()
    neg = raw.isin(VALID_NEG).sum()
    total_pos += pos; total_neg += neg
    print(f'{fname}: {len(df)}행  True={pos}  False={neg}  기타={len(df)-pos-neg}')

total = total_pos + total_neg
print(f'\n합산 — True={total_pos}  False={total_neg}  Total={total}')
if total > 0:
    print(f'pos 비율: {total_pos/total:.1%}  (pos_weight 자동 계산됩니다)')

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


class BinaryDataset(Dataset):
    def __init__(self, titles, snippets, labels, tokenizer):
        self.encodings = tokenizer(
            titles, snippets,
            truncation='only_second', padding='max_length',
            max_length=MAX_LENGTH, return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def load_data():
    frames = []
    for fname in ['labeled_titles.csv', 'labeled_players.csv']:
        path = Path(DATA_DIR) / fname
        if not path.exists():
            print(f'  SKIP: {fname} 없음')
            continue
        df = pd.read_csv(path, encoding='utf-8-sig')
        df = df.dropna(subset=['is_lotte_related']).copy()
        frames.append(df)
        print(f'  {fname}: {len(df)}행 로드')

    df = pd.concat(frames, ignore_index=True)
    df['title'] = df['title'].fillna('').astype(str).str.strip()
    df['description_snippet'] = (
        df['description_snippet'].fillna('').astype(str)
        .str[:SNIPPET_LEN].str.strip()
    )

    raw = df['is_lotte_related'].astype(str).str.strip().str.lower()
    valid_mask = raw.isin(VALID_POS | VALID_NEG)
    if not valid_mask.all():
        print(f'  DROP {(~valid_mask).sum()}행 (유효하지 않은 값)')
        df = df[valid_mask].reset_index(drop=True)
        raw = raw[valid_mask].reset_index(drop=True)

    # gpt_missing_index 행 필터링 (False 샘플에만 적용)
    # GPT 배치 처리 중 누락된 인덱스를 강제로 False 레이블한 불확실 행
    if FILTER_GPT_MISSING and 'confidence_note' in df.columns:
        is_false = raw.isin(VALID_NEG)
        is_gpt_missing = df['confidence_note'].astype(str).str.contains('gpt_missing_index', na=False)
        drop_mask = is_false & is_gpt_missing
        dropped = drop_mask.sum()
        if dropped:
            df = df[~drop_mask].reset_index(drop=True)
            raw = raw[~drop_mask].reset_index(drop=True)
            print(f'  DROP {dropped}행 (gpt_missing_index False 샘플 제거)')

    df = df.assign(_raw=raw.values)
    conflicts = df.groupby('title')['_raw'].nunique()
    conflicts = conflicts[conflicts > 1].index
    if len(conflicts):
        print(f'  DROP {len(conflicts)}개 충돌 타이틀 (동일 제목 다른 레이블)')
        df = df[~df['title'].isin(conflicts)].reset_index(drop=True)

    dedup_cols = ['title', 'source_name'] if 'source_name' in df.columns else ['title']
    df = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

    labels = df['_raw'].map(lambda v: 1.0 if v in VALID_POS else 0.0).tolist()
    pos = int(sum(labels)); neg = len(labels) - pos
    print(f'\n최종 데이터셋: {len(labels)}행  True={pos} ({pos/len(labels)*100:.1f}%)  False={neg} ({neg/len(labels)*100:.1f}%)')
    return df['title'].tolist(), df['description_snippet'].tolist(), labels


def find_threshold(y_true, probs):
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_true, probs)
    valid = [
        (float(t), float(r), float(p))
        for t, r, p in zip(thresholds, recall_arr[:-1], precision_arr[:-1])
        if r >= RECALL_TARGET
    ]
    if not valid:
        best_idx = int(np.argmax(recall_arr[:-1]))
        t = float(thresholds[best_idx])
        print(f'  [WARN] recall {RECALL_TARGET} 달성 불가 — t={t:.3f} 사용')
        return t
    best = max(valid, key=lambda x: x[2])
    print(f'  t={best[0]:.3f}  recall={best[1]:.4f}  precision={best[2]:.4f}')
    return best[0]


print('클래스 및 함수 정의 완료')

In [ ]:
def train():
    torch.manual_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_gpu = device.type == 'cuda'
    print(f'Device: {device}\n')

    titles, snippets, labels = load_data()

    tr_t, va_t, tr_s, va_s, tr_l, va_l = train_test_split(
        titles, snippets, labels,
        test_size=VAL_SPLIT, random_state=SEED, stratify=labels,
    )
    print(f'Train: {len(tr_t)}  Val: {len(va_t)}\n')

    print(f'모델 로드: {PRETRAINED} ...')
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED, num_labels=1,
    ).to(device)

    nw = 2 if use_gpu else 0
    train_ds = BinaryDataset(tr_t, tr_s, tr_l, tokenizer)
    val_ds   = BinaryDataset(va_t, va_s, va_l, tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=nw, pin_memory=use_gpu)
    val_loader   = DataLoader(val_ds,   batch_size=16,         shuffle=False, num_workers=nw, pin_memory=use_gpu)

    # pos_weight: True(롯데 관련) 클래스에 부여할 손실 가중치
    # True가 다수 클래스(~85%)이므로 raw비율(neg/pos)은 0.18 수준
    # clamp(0.3, 5.0): 너무 낮은 값은 True 클래스를 과소평가해 recall 저하 위험
    # 0.3으로 완화하여 hard negative 학습과 recall 균형 유지
    pos_sum = sum(tr_l); neg_sum = len(tr_l) - pos_sum
    pos_w = torch.tensor([neg_sum / max(pos_sum, 1)], dtype=torch.float).clamp(0.3, 5.0).to(device)
    print(f'pos_weight (raw={neg_sum/max(pos_sum,1):.3f}, clamped): {pos_w.item():.3f}')
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps_per_epoch = max(len(train_loader) // GRAD_ACCUM, 1)
    total_steps = steps_per_epoch * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps,
    )

    best = {'recall': 0.0, 'precision': 0.0, 'threshold': DEFAULT_THRESH, 'epoch': 0}
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader, 1):
            lbl = batch.pop('labels').unsqueeze(1).to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = loss_fn(model(**batch).logits, lbl) / GRAD_ACCUM
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM
            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

        model.eval()
        all_probs, all_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                lbl = batch.pop('labels').to(device)
                batch = {k: v.to(device) for k, v in batch.items()}
                all_probs.extend(torch.sigmoid(model(**batch).logits[:, 0]).cpu().tolist())
                all_true.extend(lbl.cpu().int().tolist())

        probs_arr = np.array(all_probs); true_arr = np.array(all_true)
        t = find_threshold(true_arr, probs_arr)
        preds = (probs_arr >= t).astype(int)
        r = recall_score(true_arr, preds, zero_division=0)
        p = precision_score(true_arr, preds, zero_division=0)
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  recall={r:.4f}  precision={p:.4f}')

        is_better = (
            best['epoch'] == 0
            or (r >= RECALL_TARGET and (best['recall'] < RECALL_TARGET or p > best['precision']))
            or (r > best['recall'] and best['recall'] < RECALL_TARGET)
        )
        if is_better:
            best.update(recall=r, precision=p, threshold=t, epoch=epoch)
            model.save_pretrained(OUTPUT_DIR)
            tokenizer.save_pretrained(OUTPUT_DIR)
            with open(f'{OUTPUT_DIR}/threshold.json', 'w') as f:
                json.dump({'threshold': round(t, 4)}, f, indent=2)
            print('  → Best checkpoint 저장')

    print(f'\n학습 완료 — recall={best["recall"]:.4f}  precision={best["precision"]:.4f}  t={best["threshold"]:.4f}  (epoch {best["epoch"]})')
    return best


best = train()

In [ ]:
print('=== 저장된 파일 ===')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f'  {f:<40} {size:>10,} bytes')

with open(f'{OUTPUT_DIR}/threshold.json') as f:
    thresh = json.load(f)['threshold']
print(f'\nthreshold: {thresh}')
status = '✓ 달성' if best['recall'] >= RECALL_TARGET else '✗ 미달 — epoch 증가 또는 lr 조정 고려'
print(f'recall_target={RECALL_TARGET}  {status}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
_mdl = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).eval()
THRESH = json.load(open(f'{OUTPUT_DIR}/threshold.json'))['threshold']

def predict(title, snippet=''):
    enc = _tok(title, snippet[:SNIPPET_LEN], truncation='only_second',
               padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
    with torch.no_grad():
        prob = float(torch.sigmoid(_mdl(**enc).logits[0, 0]))
    return {'related': prob >= THRESH, 'prob': round(prob, 4)}

# ── True여야 할 케이스 ───────────────────────────────────────────────────────
TRUE_CASES = [
    # 기본 롯데 자이언츠 기사
    ('롯데 나균안, 시즌 5승…선발 로테이션 안정화', '나균안이 두산전 6이닝 2실점 호투로 시즌 5승을 달성했다.'),
    ('롯데 전준우 햄스트링 부상, 2주 결장', '전준우가 1군 엔트리에서 말소됐다.'),
    ('사직구장 개막전 팬 3만 명 몰려', '롯데 자이언츠 홈 개막전 이벤트 성황리에 마무리됐다.'),
    ('롯데, 외국인 투수 교체 결정', '롯데가 부진한 외국인 투수를 방출하고 새 용병을 물색 중이다.'),
    # 2차 핵심: 1차 FP 패턴 — 경기 참여 기사 (타팀 주체여도 롯데 경기면 True)
    ('롯데 김태형 감독, 통산 800승 달성', '롯데 자이언츠 김태형 감독이 한국 역대 7번째 통산 800승 고지에 올랐다.'),
    ('KIA, 무너진 집중력에 롯데에 3:8 패', '실책에 주루사까지 겹친 KIA가 롯데에 완패했다. 롯데 선발 김진욱이 7이닝 호투.'),
    ('롯데 쿄야마, 1군 복귀 임박', '2군에서 조정을 마친 쿄야마가 이번 주 1군 합류가 예상된다.'),
    ('[롯데 관전평] 김진욱 승·최준용 세이브', '사직 홈경기에서 롯데가 삼성을 꺾고 3연승을 달렸다.'),
    ('한준수 끝내기 희생플라이, KIA 롯데 5-4로 제압', 'KIA가 롯데를 꺾고 3연패에서 탈출했다. 손성빈 실책이 결정적이었다.'),
]

# ── False여야 할 케이스 ──────────────────────────────────────────────────────
FALSE_CASES = [
    # 롯데 그룹사 (야구 무관)
    ('롯데백화점, 봄 세일 시작', '롯데백화점이 봄 할인 행사를 시작했다.'),
    ('롯데월드, 신규 어트랙션 공개', '롯데월드가 여름 신규 놀이기구를 선보였다.'),
    ('롯데칠성음료, 온실가스 6400톤 감축', '롯데칠성이 2040 탄소중립 목표를 향해 온실가스를 감축했다.'),
    # 타팀 단독 기사 (롯데 언급 없음)
    ('삼성 라이온즈, KIA 잡고 선두 탈환', 'KIA 타이거즈가 삼성에 패해 2위로 내려앉았다.'),
    # 2차 핵심: 타팀 분석 기사 (롯데는 상대·비교 대상으로만 잠깐 등장)
    ('KIA 양창섭, 대롯데 완봉…삼성 선발진 기둥으로', '양창섭이 삼성의 선발 에이스로 인정받았다. 롯데 타선을 상대로 완봉승.'),
    ('차기 여신금융협회장에 이동철 전 KB금융 부회장 내정', '여신금융협회가 이동철 후보를 차기 회장으로 추천했다.'),
    # 4차 핵심: 라운드업형 — 여러 팀 경기 묶음, 롯데는 snippet에만 등장
    ("'박준현 데뷔전 선발승' 키움, 삼성과 3연전 스윕…KT는 선두 탈환",
     '광주에서는 롯데 자이언츠와 KIA 타이거즈가 연장 혈투 끝에 5-5로 승부를 가리지 못했다.'),
    # 4차 핵심: 타팀 선수 이슈 분석 — 롯데 경기가 부상/이슈의 배경으로만 언급
    ('KIA 카스트로 햄스트링 파열, 전력 이탈 대체 외인 물색',
     '카스트로는 롯데 자이언츠와의 경기 중 수비하다 허벅지 통증을 호소해 교체됐다.'),
]

print(f'Threshold: {THRESH}\n')
ok = 0; total = len(TRUE_CASES) + len(FALSE_CASES)
print('=== True여야 할 케이스 ===')
for title, snippet in TRUE_CASES:
    r = predict(title, snippet)
    mark = 'O' if r['related'] else 'X'
    ok += int(r['related'])
    print(f'  {mark} prob={r["prob"]:.4f}  {title}')
print('\n=== False여야 할 케이스 ===')
for title, snippet in FALSE_CASES:
    r = predict(title, snippet)
    mark = 'O' if not r['related'] else 'X'
    ok += int(not r['related'])
    print(f'  {mark} prob={r["prob"]:.4f}  {title}')
print(f'\n결과: {ok}/{total} 통과')
if ok < total:
    print('[WARN] 실패 케이스 확인 후 threshold 재조정 또는 추가 epoch 검토')

In [ ]:
import shutil, zipfile
from google.colab import files

ZIP = '/content/lotte_related_model.zip'
shutil.make_archive('/content/lotte_related_model', 'zip', OUTPUT_DIR)
print(f'압축 완료: {ZIP}')
with zipfile.ZipFile(ZIP) as z:
    for name in sorted(z.namelist()):
        print(f'  {name:<45} {z.getinfo(name).file_size:>10,} bytes')
files.download(ZIP)

## 다운로드 후 로컬 배치

```
lotte_related_model.zip 압축 해제
  → training/models/lotte_related_koelectra/
```

필수 파일:
- `config.json`, `model.safetensors` (또는 `pytorch_model.bin`)
- `tokenizer_config.json`, `vocab.txt`
- `threshold.json` ← 없으면 IS_LOTTE_RELATED_THRESHOLD(0.40) 사용

배치 완료 후 `backend/models/lotte_related_detector.py`가 자동으로 모델을 로드합니다.